### Data Ingestion

In [23]:
### Document DataStructure 

from langchain_core.documents import Document 

In [24]:
doc=Document(page_content="This is the main text content I am using to create RAG.", 
metadata={
    "source": "sample.txt",
    "pages": 10,
    "author": "MehiManswa"
    }
)
doc

Document(metadata={'source': 'sample.txt', 'pages': 10, 'author': 'MehiManswa'}, page_content='This is the main text content I am using to create RAG.')

In [25]:
### Create a simple txt file

import os
os.makedirs("../data/text_files", exist_ok=True)

In [26]:
sample_texts={
"../data/text_files/python_intro.txt": """Python Programming Introduction
Python is a high-level, interpreted programming language known for its simplicity and readability.
Created by Guido van Rossum and first released in 1991, Python has become one of the most popular programming Languages in the world.
Key Features:
- Easy to learn and use
- Extensive standard library
- Cross-platform compatibility
Strong community support

Python is widely used in web development, data science, artificial intelligence, and automation. "
""",}

sample_texts={
"../data/text_files/machine_learning.txt": """Machine Learning Basics
Machine learning is a subset of artificial intelligence that enables systems to learn and improve from experience without being explicitly programmed. It focuses on developing computer programs that can access data and use it to learn for themselves.
Types of Machine Learning:
1. Supervised Learning: Learning with labeled data
2. Unsupervised Learning: Finding patterns in unlabeled data
3. Reinforcement Learning: Learning through rewards and penalties
Applications include image recognition, speech processing, and recommendation systems
""",}
for filepath, content in sample_texts.items(): 
    with open(filepath, 'w',encoding="utf-8") as f:
        f.write(content)
print('Sample text files created successfully.')

Sample text files created successfully.


In [27]:
### TextLoader
from langchain_community.document_loaders import TextLoader
loader=TextLoader("../data/text_files/python_intro.txt", encoding="utf-8")
documents=loader.load()
print(documents)

[Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content='Python Programming Introduction\nPython is a high-level, interpreted programming language known for its simplicity and readability.\nCreated by Guido van Rossum and first released in 1991, Python has become one of the most popular programming Languages in the world.\nKey Features:\n- Easy to learn and use\n- Extensive standard library\n- Cross-platform compatibility\nStrong community support\n\nPython is widely used in web development, data science, artificial intelligence, and automation. "\ndata/text_files/machine_learning.txt": **Machine Learning Basics\nMachine learning is a subset of artificial intelligence that enables systems to learn and improve from experience without being explicitly programmed. It focuses on developing computer programs that can access data and use it to learn for themselves.\nTypes of Machine Learning:\n1. Supervised Learning: Learning with labeled data\n2. Unsupervised Lear

In [28]:
### Directory Loader
from langchain_community.document_loaders import DirectoryLoader, TextLoader

## load all the text files from the directory 
dir_loader = DirectoryLoader(
    "../data/text_files",
    glob="**/*.txt", ## Fixed: Added the missing dot for .txt files
    loader_cls=TextLoader, ## loader class to use
    loader_kwargs={"encoding": "utf-8"}, ## Fixed: Matched the double quotes perfectly
    show_progress=False
)

documents = dir_loader.load()
documents

[Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content='Python Programming Introduction\nPython is a high-level, interpreted programming language known for its simplicity and readability.\nCreated by Guido van Rossum and first released in 1991, Python has become one of the most popular programming Languages in the world.\nKey Features:\n- Easy to learn and use\n- Extensive standard library\n- Cross-platform compatibility\nStrong community support\n\nPython is widely used in web development, data science, artificial intelligence, and automation. "\ndata/text_files/machine_learning.txt": **Machine Learning Basics\nMachine learning is a subset of artificial intelligence that enables systems to learn and improve from experience without being explicitly programmed. It focuses on developing computer programs that can access data and use it to learn for themselves.\nTypes of Machine Learning:\n1. Supervised Learning: Learning with labeled data\n2. Unsupervised Lear

In [29]:
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_community.document_loaders import DirectoryLoader

# Load all the PDF files from the directory
dir_loader = DirectoryLoader(
    "../data/MACHINE LEARNING",
    glob="**/*.pdf",
    loader_cls=PyMuPDFLoader, # type: ignore
    show_progress=False
)

pdf_documents = dir_loader.load()
print(pdf_documents)

[Document(metadata={'producer': '', 'creator': '', 'creationdate': '', 'source': '../data/MACHINE LEARNING/Artificial_Intelligence__Sem-IV__Batch_2021-22__2022-23__Re_Exam_DcOLEZkJOF.pdf', 'file_path': '../data/MACHINE LEARNING/Artificial_Intelligence__Sem-IV__Batch_2021-22__2022-23__Re_Exam_DcOLEZkJOF.pdf', 'total_pages': 6, 'format': 'PDF 1.2', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': '', 'page': 0}, page_content=''), Document(metadata={'producer': '', 'creator': '', 'creationdate': '', 'source': '../data/MACHINE LEARNING/Artificial_Intelligence__Sem-IV__Batch_2021-22__2022-23__Re_Exam_DcOLEZkJOF.pdf', 'file_path': '../data/MACHINE LEARNING/Artificial_Intelligence__Sem-IV__Batch_2021-22__2022-23__Re_Exam_DcOLEZkJOF.pdf', 'total_pages': 6, 'format': 'PDF 1.2', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': '', 'page': 1}, page_content=''

In [30]:
type(pdf_documents[0])

langchain_core.documents.base.Document

### embedding And vectorStoreDB

In [31]:
import numpy as np 
from sentence_transformers import SentenceTransformer 
import chromadb
from chromadb.config import Settings 
import uuid
from typing import List,Dict,Any,Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [32]:
from langchain_core import embeddings


class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
                """
                Initialize the embedding manager
                Args:
                    model_name: HuggingFace model name for sentence embeddings
                """
                self.model_name = model_name
                self.model = None
                self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print (f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model: {e}")

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        Args:
            texts: List of text strings
        Returns:
            Numpy array of embeddings
        """        
        if not self.model:
            raise ValueError("Model not loaded. Cannot generate embeddings.")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True, convert_to_numpy=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings
    
## initialize the
embedding_manager=EmbeddingManager()
embedding_manager



Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5075.17it/s]


Model loaded successfully. Embedding dimension: 384


### VectorStore

In [33]:
import os
import uuid
import numpy as np
import chromadb
from typing import List, Any

class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""

    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )

            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """Add documents and their embeddings to the vector store"""
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")

        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)
            metadatas.append(metadata)

            documents_text.append(doc.page_content)
            embeddings_list.append(embedding.tolist())

        try:
            if self.collection is None:
                raise ValueError("Collection not initialized.")
            
            self.collection.add(
                ids=ids,
                metadatas=metadatas,
                documents=documents_text,
                embeddings=embeddings_list
            )
            print(f"Successfully added {len(documents)} documents to vector store.")
        except Exception as e:
            print(f"Error adding documents: {e}")
            raise

    def search_similar(self, query_embedding: np.ndarray, n_results: int = 4):
        """
        Search the ChromaDB collection for the most similar documents.
        """
        if self.collection is None:
            raise ValueError("Collection not initialized.")
            
        # Failsafe: if the database is empty, return early
        if self.collection.count() == 0:
            print("Warning: The database is empty! Please run add_documents first.")
            return None

        print(f"Searching for top {n_results} closest matches...")
        
        try:
            # ChromaDB expects a list of lists for query embeddings
            results = self.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=n_results
            )
            return results
        except Exception as e:
            print(f"Error during search: {e}")
            return None

In [34]:
def split_documents(documents, chunk_size=1000, overlap=200):
    chunks = []
    for doc in documents:
        text = doc.page_content or ""
        if not text.strip():
            continue
        start = 0
        while start < len(text):
            end = min(start + chunk_size, len(text))
            chunk_text = text[start:end]
            metadata = dict(doc.metadata)
            metadata["chunk_start"] = start
            metadata["chunk_end"] = end
            chunks.append(Document(page_content=chunk_text, metadata=metadata))
            if end == len(text):
                break
            start += chunk_size - overlap
    return chunks

chunks = split_documents(pdf_documents)
chunks

[Document(metadata={'producer': 'Adobe Acrobat 26 Paper Capture Plug-in', 'creator': 'hp officejetpro', 'creationdate': '2025-04-28T11:05:03+05:30', 'source': '../data/MACHINE LEARNING/Machine_Learning__Semester_IV__Final_Exam_2024-25_and_Re_Exam_2022-23_and_2023-24_wDzRy8Ph38.pdf', 'file_path': '../data/MACHINE LEARNING/Machine_Learning__Semester_IV__Final_Exam_2024-25_and_Re_Exam_2022-23_and_2023-24_wDzRy8Ph38.pdf', 'total_pages': 6, 'format': 'PDF 1.6', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2026-04-25T03:45:20+05:30', 'trapped': '', 'modDate': "D:20260425034520+05'30'", 'creationDate': "D:20250428110503+05'30'", 'page': 0, 'chunk_start': 0, 'chunk_end': 1000}, page_content='. \n\'•· \nSVKM\'S NMIMS \nMUKESH PATEL SCHOOL OF TECHNOLOGY MANAGEMENT& ENGINEERING/ \nSCHOOL OF TECHNOLOGY MANAGEMENT \nAcademic Year: 2024-2025 \nProgram/s: BTech/MBA Tech \nStream/s: AI/ AIML/ AIDS/CSE(DS) . \nSubject: Machine Leaming \nYear: II Semester: IV \nTime: 03 hrs (1o;

In [35]:
### Convert the text to embeddings 
texts = [doc.page_content for doc in chunks]

## Generate the Embeddings
embeddings = embedding_manager.generate_embeddings(texts)

## Initialize the database (if you haven't already in this specific cell)
vectorstore = VectorStore()

## Store in the vector database using the INSTANCE (lowercase 'v')
vectorstore.add_documents(chunks, embeddings)

Generating embeddings for 17 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  3.37it/s]

Generated embeddings with shape: (17, 384)
Vector store initialized. Collection: pdf_documents
Existing documents in collection: 170
Adding 17 documents to vector store...
Successfully added 17 documents to vector store.


In [36]:
#Checking if the documents are added successfully
print(f"Total chunks ready to save: {len(chunks)}")

### Convert the text to embeddings 
texts = [doc.page_content for doc in chunks]
# ... [rest of your code] ...

Total chunks ready to save: 17


### Retriever Pipeline From

In [37]:
from typing import List, Dict, Any

class RAGRetriever:
    """Retrieves relevant documents from the vector store based on query embeddings"""

    def __init__(self, embedding_manager, vector_store):
        self.embedding_manager = embedding_manager
        self.vector_store = vector_store

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.1) -> List[Dict[str, Any]]:
        """Retrieve relevant documents for a given query"""
        
        # 1. Vectorize the query
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # 2. Query the database
        results = self.vector_store.collection.query(
            query_embeddings=[query_embedding.tolist()],
            n_results=top_k,
        )
        
        retrieved_docs = []

        if results and results.get("documents") and results["documents"][0]:
            documents = results["documents"][0] 
            metadatas = results.get("metadatas", [[]])[0] 
            distances = results.get("distances", [[]])[0] 
            ids = results.get("ids", [[]])[0]

            # 3. Process results (Removed the redundant double-loop)
            for i, (doc_id, document, metadata, distance) in enumerate(
                zip(ids, documents, metadatas, distances)
            ):
                # THE FIX: Safely convert L2 Distance (0 to 2+) into a 0-1 Similarity Percentage
                # We use 1 / (1 + distance) as a robust normalization that never goes negative
                similarity_score = 1.0 / (1.0 + distance) 
                
                # Check against the threshold
                if similarity_score >= score_threshold:
                    retrieved_docs.append({
                        "id": doc_id,
                        "content": document,
                        "metadata": metadata,
                        "similarity_score": similarity_score,
                        "distance": distance,
                        "rank": i + 1,
                    })

            print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
        else:
            print("No documents found in database.")

        return retrieved_docs

# Initialize with the fixed class
retriever = RAGRetriever(embedding_manager, vectorstore)

In [38]:
retriever.retrieve(
    "Explain the term curse of dimensionality. What problem does it cause? List down the methods used in Dimensionality Reduction?",
    top_k=3,
    score_threshold=0.1
)

Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 36.22it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


[{'id': 'doc_9177da26_16',
  'content': " \n144 \n445000 \n5. \n23000 \n.37 \n600000 \n6.\n35550\n350 \n350000 \n7.\n42000\n45 \n450000 \n8.\n45000\n112 \n550000 \n9.\n27700\n77 \n250000 \n10.\n12345\n82 \n377133 \nExplain the term 'curse of dimensionality'. What problem does it cause? List down the \nniethods used in·Dimensionality Reduction. \n1.\nUse the K nearest neighbor classifier (KNN) to categorize the patient with\nBMI=43.6 and Age=40 as diabetic/non-diabetic. Given the training examples:\n(Assume K =3 and use Euclidean Distance)\n2.\nWhat will be the classification if k=5? How do we choose the optimum value of K\nin KNN algoi-ithm?\nBMI \nAge \nDiabetic/non-\ndiabetic \n33.6. \n50 \n1 \n26.6 \n30 \n0 \n23.4 \n40 \n0 \n43.1 \n67 \n0 \n35.3 \n23 \n1 \n35.9' \n'67 \n1 \nt '\n[5] \n' ' \n[5] \n[10]",
  'metadata': {'doc_index': 16,
   'title': '',
   'total_pages': 5,
   'trapped': '',
   'page': 1,
   'subject': '',
   'source': '../data/MACHINE LEARNING/Machine_Learning__Sem-IV

### Integration of VectorDB with LLM output


In [39]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq # type: ignore

# Load environment variables (make sure GROQ_API_KEY is in your .env file)
load_dotenv()

### Initialize the GROQ LLM
# Fetch the key securely from the environment rather than hardcoding it
groq_api_key = os.getenv("GROQ_API_KEY")
if not groq_api_key or groq_api_key.strip() == "":
    raise RuntimeError(
        'GROQ_API_KEY is not set. Add it to .env or export GROQ_API_KEY="gsk-..." before running.'
    )

groq_api_key = groq_api_key.strip()

# Initialize the GROQ LLM with a currently supported model
llm = ChatGroq(
    groq_api_key=groq_api_key,  # type: ignore
    temperature=0.7, 
    max_tokens=1024, 
    model="llama-3.1-8b-instant" # <--- Changed from gemma2-9b-it
)

# Simple RAG function
# Note: Ensure 'RAGRetriever' is imported or defined elsewhere in your script
def rag_pipeline(query: str, retriever, llm: ChatGroq, top_k: int = 5, score_threshold: float = 0.1) -> str:
    # 1. Fetch the documents
    retrieved_docs = retriever.retrieve(query, top_k=top_k, score_threshold=score_threshold)
    
    if not retrieved_docs:
        return "No relevant documents found to answer the query."

    # 2. Build the context string
    context = "\n\n".join([f"Document {doc['rank']} (Score: {doc['similarity_score']:.2f}):\n{doc['content']}" for doc in retrieved_docs])
    
    # 3. Create the prompt using an f-string (Formatting happens here automatically)
    prompt = f"""Use the following retrieved documents to answer the question:

{context}

Question: {query}
Answer:"""
    
    # 4. Invoke the LLM by passing the already-formatted string
    response = llm.invoke(prompt)
    
    return response.content  # type: ignore

In [40]:
answer = rag_pipeline(
    query="Explain the term curse of dimensionality. What problem does it cause? List down the methods used in Dimensionality Reduction?",
    retriever=retriever, # Make sure your 'retriever' object is initialized before this line!
    llm=llm,
    top_k=3,
    score_threshold=0.1
)

print(answer)

Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 78.09it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


The curse of dimensionality is a term used in the field of machine learning and data analysis to describe the exponential increase in the number of data points in high-dimensional spaces. As the number of features or dimensions in a dataset increases, the volume of the data space grows exponentially, making it increasingly difficult to analyze and work with.

The problems caused by the curse of dimensionality include:

1. **Increased noise**: As the dimensionality of the data increases, the noise in the data becomes more dominant, making it harder to identify meaningful patterns.
2. **Decreased data density**: In high-dimensional spaces, data points become increasingly sparse, making it harder to find meaningful relationships between features.
3. **Computational complexity**: Many machine learning algorithms become computationally expensive or even infeasible to run in high-dimensional spaces.

To mitigate these problems, dimensionality reduction techniques are used to reduce the numbe

In [41]:
answer = rag_pipeline(
    query="Explain  how  Regularization techniques improve machine learning models by preventing overfitting. Also, compare Ridge  Regression and Lasso 1;  BL-3 Regression techniques of  regularization.",
    retriever=retriever, # Make sure your 'retriever' object is initialized before this line!
    llm=llm,
    top_k=3,
    score_threshold=0.1
)

print(answer)

Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 16.83it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


Regularization techniques are used in machine learning to prevent overfitting, which occurs when a model is too complex and performs well on the training data but poorly on new, unseen data. Overfitting can be caused by a model having too many parameters or being too complex.

Regularization techniques work by adding a penalty term to the loss function, which discourages the model from making large changes to the parameters. This encourages the model to find a simpler solution that generalizes better to new data.

There are several types of regularization techniques, including:

1. **L1 Regularization (Lasso Regression)**: This technique adds a term to the loss function that is proportional to the absolute value of the parameters. This encourages the model to set some parameters to zero, effectively selecting the most important features.
2. **L2 Regularization (Ridge Regression)**: This technique adds a term to the loss function that is proportional to the square of the parameters. Thi

### Enhanced RAG PipeLine Features

In [42]:
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.1, return_content=False):
    """
    RAG pipeline with extra features like:
    - Returns answers, confidence scores, sources and full context.
    """
    # 1. Retrieve the documents
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    
    if not results:
        return {"answer": "No relevant documents found.", "sources": [], "confidence": 0, "context": []}
    
    # 2. Build context and sources list
    context = "\n\n".join([f"Document {doc['rank']} (Score: {doc['similarity_score']:.2f}):\n{doc['content']}" for doc in results])
    
    sources = [{
        'source': doc['metadata'].get('source', 'unknown'), # Fixed typo from 'sourece'
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:100] + '...',
    } for doc in results]
    
    confidence = max([doc['similarity_score'] for doc in results])

    # 3. Define the prompt (This was missing!)
    prompt = f"""Use the following retrieved documents to answer the question:
    
{context}

Question: {query}
Answer:"""

    # 4. Generate the answer using the LLM (No need for .format() if using an f-string directly above)
    response = llm.invoke(prompt)  
    answer = response.content  

    # 5. Build the output dictionary (Fixed the response.answer bug)
    output = {
        "answer": answer, 
        "sources": sources,
        "confidence": confidence,
    }
    
    if return_content:
        output["context"] = context
        
    return output

# --- Execution ---
result = rag_advanced(
    query="Illustrate the  difference between MCAR, MAR,  and  MNAR  in  missing  data mechanisms. Discuss in detail.", 
    retriever=retriever, 
    llm=llm, 
    top_k=5, 
    min_score=0.1, 
    return_content=True
)

print('Answer:', result.get('answer'))
print('Sources:', result.get('sources'))
print('Confidence:', result.get('confidence'))
print('Context:', result.get('context'))

Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 17.06it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


Answer: The question is present in multiple documents with the same description, so I'll refer to the content as a single entity.

Missing data mechanisms are classified into three main categories: Missing Completely at Random (MCAR), Missing at Random (MAR), and Missing Not at Random (MNAR). 

1. **Missing Completely at Random (MCAR)**: 
In MCAR, the probability of missingness does not depend on either observed or unobserved data. In other words, the data is missing randomly and does not follow any pattern. For example, flipping a coin to determine whether to record data or not. This is the most ideal scenario, as the missing data do not affect the analysis.

2. **Missing at Random (MAR)**: 
In MAR, the probability of missingness depends on some observed data, but not on the missing data itself. This means that the missing data may be related to some observed variables, but not to the variable that is missing. For instance, suppose a researcher wants to study the relationship between 

In [43]:
result = rag_advanced(
    query="Describe  the  importance of  feature selection.  Discuss the  various  feature selection methods  to  select  the  most  relevant  features. ", 
    retriever=retriever, 
    llm=llm, 
    top_k=5, 
    min_score=0.1, 
    return_content=True
)

print('Answer:', result.get('answer'))
print('Sources:', result.get('sources'))
print('Confidence:', result.get('confidence'))
print('Context:', result.get('context'))

Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 14.55it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


Answer: The importance of feature selection lies in the ability of a model to improve its accuracy, reduce overfitting, and enhance interpretability by selecting the most relevant features from a large dataset. This is particularly crucial in machine learning, as irrelevant or redundant features can lead to decreased model performance, increased computational costs, and poor generalization to unseen data.

Feature selection methods can be broadly categorized into three types: filter methods, wrapper methods, and embedded methods.

**Filter Methods:**

1. **Chi-Squared Test:** Measures the independence between a feature and the target variable.
2. **Mutual Information:** Estimates the mutual information between a feature and the target variable.
3. **Correlation Analysis:** Measures the correlation between features and the target variable.

**Wrapper Methods:**

1. **Recursive Feature Elimination (RFE):** Selects features recursively by iteratively removing the least important features.

### Checking the distance

In [44]:
# 1. Vectorize the exact question using your embedding manager
test_query = "Illustrate the  difference between MCAR, MAR,  and  MNAR  in  missing  data mechanisms. Discuss in detail. "
query_vector = embedding_manager.generate_embeddings([test_query])[0]

# 2. Query the ChromaDB collection directly (bypassing all threshold filters)
raw_results = vectorstore.collection.query( # type: ignore
    query_embeddings=[query_vector.tolist()],
    n_results=3
)

# 3. Print what the database actually holds
print("--- DIRECT DATABASE RESULTS ---")
print(f"Distances: {raw_results['distances']}")
print(f"Documents Found: {len(raw_results['documents'][0])}") # type: ignore

Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  3.45it/s]

Generated embeddings with shape: (1, 384)
--- DIRECT DATABASE RESULTS ---
Distances: [[1.220870018005371, 1.220870018005371, 1.220870018005371]]
Documents Found: 3
